## ISOM4030 Lab 6.2 - Model Tuning

### Overview

The process of selecting the right set of hyperparameters for your machine learning (ML) application is called *hyperparameter tuning* or *hypertuning*.

Hyperparameters are the variables that govern the training process and the topology of an ML model. These variables remain constant over the training process and directly impact the performance of your ML program.

In this lab, we will introduce various model tuning techniques, including batch normalization, dropout layers, optimizers with momentum, and learning rate decay schedules. These techniques are fundamental for optimizing and improving the performance of machine learning models.

Additionally, we will explore the Keras Tuner, a powerful tool that assists in selecting the optimal set of hyperparameters for your TensorFlow program. Through this lab, you will gain practical insights into these tuning techniques and learn how to leverage the Keras Tuner effectively.



#### Setup

In [38]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, losses, optimizers, callbacks
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization

Install and import the Keras Tuner.

In [39]:
!pip install -q -U keras-tuner # -q is short for quiet; -U is short for upgrade

In [40]:
import keras_tuner as kt

#### Download and prepare the dataset

In this tutorial, you will use the Keras Tuner to find the best hyperparameters for a machine learning model that classifies images of clothing from the [Fashion MNIST dataset](https://github.com/zalandoresearch/fashion-mnist).

In [41]:
(img_train, label_train), (img_test, label_test) = keras.datasets.fashion_mnist.load_data()

In [42]:
# normalize pixel values between 0 and 1
img_train = img_train.astype('float32') / 255.0
img_test = img_test.astype('float32') / 255.0

#### Warm up: Build the model with Sequential API.

In [43]:
# Model built with Sequential API

# Step 1: Build the model (Define the layers and stack them into a network in order)
model_init = Sequential([
                    layers.Flatten(input_shape=(28, 28)),
                    layers.Dense(128, activation='relu'),
                    layers.Dense(10)
                ])
# Print a summary of the network to examine model architecture
print(model_init.summary())


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_3 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)

None


### BatchNormalization Layer
A Batch Normalization (BN) layer is implemented as [`tf.keras.layers.BatchNormalization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/BatchNormalization)

In a BN layer, there are 4 parameters for each node in the previous dense layer: `gamma`, `beta`, `moving_mean`, and `moving_variance`.

`Gamma (scale)` and `beta (offset)` are trainable parameters. `moving_mean` and `moving_variance` are non-trainable parameters (used during prediction).

With a dense layer of 128 nodes:

- Trainable parameters: 128 (gamma) + 128 (beta) = 256
- Non-trainable parameters: 128 (moving_mean) + 128 (moving_variance) = 256

Total parameters in BN layer: 256 (trainable) + 256 (non-trainable) = 512


Benefits of BN layer:
- Improve training stability, allow higher learning rate
- Accelerate training convergence, enhance gradient flow
- Regularization Effect

<font color = 'red'><b>Note</b></font>: There are two opinions for where the BN layer should be placed in the model architecture — before and after activation.

In [44]:
# Add BN layer after activation function
model_init = Sequential([
            layers.Flatten(input_shape=(28, 28)),
            layers.Dense(128, activation='relu'),
            layers.BatchNormalization(),    # By default: trainable = True
            layers.Dense(10)
])

model_init.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_4 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,282 (399.54 KB)

 Trainable params: 102,026 (398.54 KB)

 Non-trainable params: 256 (1.00 KB)

In [45]:
# Add BN layer before activation function
model_init = Sequential([
            layers.Flatten(input_shape=(28, 28)),
            layers.Dense(128),
            layers.BatchNormalization(),    # By default: trainable = True
            layers.ReLU(),
            layers.Dense(10)
])
model_init.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_5 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,282 (399.54 KB)

 Trainable params: 102,026 (398.54 KB)

 Non-trainable params: 256 (1.00 KB)

### Dropout Layer
Last time, we mentioned the dropout layer - [`tf.keras.layers.Dropout()`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dropout). The Dropout layer randomly sets input units to 0 with a frequency of rate at each step during training time, which helps prevent overfitting. Inputs not set to 0 are scaled up by 1/(1 - rate) such that the sum over all inputs is unchanged.


<font color = 'red'><b>Note</b></font>: Do not add BN layer after dropout.


During training, dropout is applied randomly and independently for each forward pass or each epoch. That means, in each batch, a different random subset of neurons is dropped. So across epochs, the dropout pattern changes constantly. *Therefore, each neuron has a chance to be trained.* This randomness is what gives dropout its regularization power — it prevents the network from relying too heavily on any specific neuron.

During testing, all neurons are active and contribute to the output. So we have to scale up each neuron's ouput during training stage to match with the expected output during testing.

In [46]:
model_init = Sequential([
            layers.Flatten(input_shape=(28, 28)),
            layers.Dense(128),
            layers.BatchNormalization(),    # By default: trainable = True
            layers.ReLU(),
            layers.Dropout(0.2),
            layers.Dense(10)
])
model_init.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_6 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,282 (399.54 KB)

 Trainable params: 102,026 (398.54 KB)

 Non-trainable params: 256 (1.00 KB)

### Optimizer selection
SGD is a simple optimization algorithm, Adam offer adaptive learning rates and additional techniques to accelerate convergence. Adam, in particular, combines the benefits of both SGD and RMSprop, making it a popular choice in many deep learning applications.



#### Default Values - For Your Reference Only

<font color = 'red'><b>Default values in key functions</b> </font>

**SGD** vs **Adam**:

- [`SGD()` constructor](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/SGD)

```python
tf.keras.optimizers.experimental.SGD(
    learning_rate=0.01, ...
)
```


- [`Adam()` constructor](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam)

```python
tf.keras.optimizers.Adam(
    learning_rate=0.001, ...
)

```


### EarlyStopping

Create a callback to stop training early after reaching a certain value for the validation loss. Specify the `callbacks` by the defined earlystopping criteria while the model is trained with `.fit()`.

*patience=2*: This means the training will stop after 2 consecutive epochs where the validation loss does not improve.

In [47]:
stop_early = callbacks.EarlyStopping(monitor='val_loss', patience=2)

<font color = 'steelblue'><b>Exercise</b> </font>

Q1: Build a neural network model in TensorFlow with the given specifications:

- Flatten layer with input shape: (28, 28)
- Hidden layer: One hidden layer with 128 nodes, activated by ReLU
- Batch Normalization (BN) layer: Implemented for improved training stability
- Dropout layer: 20%
- Optimizer: Adam optimizer with default learning rate
- Batch size: 32
- No. of epochs: 10
- EarlyStopping callback: Included to monitor validation loss
- 20% data split on training set for evaluation

In [37]:
# Step 1: build the model
model_ex1 = Sequential([layers.Flatten(input_shape=(28, 28)),
                    layers.Dense(128),
                    layers.BatchNormalization(),    # By default: trainable = True
                    layers.ReLU(),
                    layers.Dropout(0.2),
                    layers.Dense(10)]
)

# Step 2: Compile the model (Specify the loss function and optimizer to use)
model_ex1.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Step 3: Train the model (Specify batch size and number of epochs)
model_ex_output = model_ex1.fit(img_train, label_train, batch_size=32, epochs=20, validation_split=0.2,  callbacks=[stop_early])

Epoch 1/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.7782 - loss: 0.6405 - val_accuracy: 0.8269 - val_loss: 0.4733
Epoch 2/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8524 - loss: 0.4104 - val_accuracy: 0.8422 - val_loss: 0.4259
Epoch 3/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8614 - loss: 0.3765 - val_accuracy: 0.8629 - val_loss: 0.3706
Epoch 4/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8692 - loss: 0.3506 - val_accuracy: 0.8612 - val_loss: 0.3728
Epoch 5/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8786 - loss: 0.3377 - val_accuracy: 0.8747 - val_loss: 0.3449
Epoch 6/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8842 - loss: 0.3166 - val_accuracy: 0.8765 - val_loss: 0.3396
Epoch 7/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8870 - loss: 0.3097 - val_accuracy: 0.8717 - val_loss: 0.3465
Epoch 8/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8911 - loss: 0.2952 - 

### Keras Tuner the model tuning

When you build a model for hypertuning, you also define the hyperparameter search space in addition to the model architecture. The model you set up for hypertuning can be perceived as a *hypermodel*.

You can define a hypermodel by using a model builder function, which defines inline the hyperparameters you want to tune and returns a compiled model.

In [61]:
def model_builder(hp):
    # Tune the number of units in the 1st Dense layer
    # Choose an optimal value between 32-512
    hp_units_1 = hp.Int('units_1', min_value=32, max_value=512, step=32)
    hp_units_2 = hp.Int('units_2', min_value=32, max_value=512, step=32)

    model = keras.Sequential()
    model.add(Flatten(input_shape=(28, 28)))
    model.add(Dense(units=hp_units_1, activation='relu'))

    if hp.Boolean("bn_1"):
        model.add(BatchNormalization())

    model.add(Dense(units=hp_units_2, activation='relu'))
    if hp.Boolean("bn_2"):
        model.add(BatchNormalization())
    model.add(Dense(10))

    # Tune the learning rate for the optimizer
    # Choose an optimal value from 0.01, 0.001, or 0.0001
    hp_learning_rate = hp.Float("learning_rate", min_value=0.001, max_value=10,
                              step=3, sampling="log")   # log - nonlinear: [0.001, 0.01, 0.1, 1, 10]

    hp_optimizer = hp.Choice('optimizer', values = ['adam', 'sgd'], ordered = False)

    if hp_optimizer == 'adam':
        opt_algo = optimizers.Adam(learning_rate=hp_learning_rate)
    else:
        opt_algo = optimizers.SGD(learning_rate=hp_learning_rate)

    model.compile(optimizer=opt_algo,
                    loss=losses.SparseCategoricalCrossentropy(from_logits=True),
                    metrics=['accuracy'])

    return model

you can find the way to tune the number of layers here:
https://keras.io/guides/keras_tuner/getting_started/

## Instantiate the tuner and perform hypertuning

Instantiate the tuner to perform the hypertuning. The Keras Tuner has 5 tuners available - `RandomSearch`, `Hyperband`, `BayesianOptimization`, and `Sklearn`.

In this tutorial, you use the [`RandomSearch` tuner](https://keras.io/api/keras_tuner/tuners/random/#randomsearch-class).

To instantiate the Hyperband tuner, you must specify the `hypermodel`, the `objective` to optimize, and the total number of experiments (model configurations) to test at most (`max_trials`).

In [63]:
! rm -r my_dir

In [64]:
tuner = kt.RandomSearch(model_builder,
                        objective='val_accuracy',
                        max_trials=10,
                        seed=1,                # to make sure the result is reproducible
                        directory='my_dir',   # create a folder to store the model
                        project_name='intro_to_kt')

You can print a summary of the search space:

In [65]:
tuner.search_space_summary()

Search space summary
Default search space size: 6
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
units_2 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
bn_1 (Boolean)
{'default': False, 'conditions': []}
bn_2 (Boolean)
{'default': False, 'conditions': []}
learning_rate (Float)
{'default': 0.001, 'conditions': [], 'min_value': 0.001, 'max_value': 10.0, 'step': 3, 'sampling': 'log'}
optimizer (Choice)
{'default': 'adam', 'conditions': [], 'values': ['adam', 'sgd'], 'ordered': False}


Run the hyperparameter search. The arguments for the search method are the same as those used for `tf.keras.model.fit` in addition to the callback above.

In [66]:
stop_early = callbacks.EarlyStopping(monitor='val_loss', patience=2)

In [67]:
tuner.search(img_train, label_train, batch_size=256, epochs=5, validation_split=0.2, callbacks=[stop_early])

Trial 10 Complete [00h 00m 09s]
val_accuracy: 0.8545833230018616

Best val_accuracy So Far: 0.875249981880188
Total elapsed time: 00h 01m 19s


To check the summary for the hypertuning job, we simply use `.results_summary()`:

In [68]:
tuner.results_summary(5)

Results summary
Results in my_dir/intro_to_kt
Showing 5 best trials
Objective(name="val_accuracy", direction="max")

Trial 04 summary
Hyperparameters:
units_1: 224
units_2: 384
bn_1: True
bn_2: False
learning_rate: 0.081
optimizer: sgd
Score: 0.875249981880188

Trial 01 summary
Hyperparameters:
units_1: 192
units_2: 128
bn_1: False
bn_2: True
learning_rate: 0.081
optimizer: adam
Score: 0.862416684627533

Trial 09 summary
Hyperparameters:
units_1: 64
units_2: 512
bn_1: False
bn_2: True
learning_rate: 0.009000000000000001
optimizer: adam
Score: 0.8545833230018616

Trial 07 summary
Hyperparameters:
units_1: 32
units_2: 224
bn_1: False
bn_2: True
learning_rate: 0.027
optimizer: sgd
Score: 0.8539999723434448

Trial 00 summary
Hyperparameters:
units_1: 96
units_2: 512
bn_1: False
bn_2: False
learning_rate: 0.243
optimizer: sgd
Score: 0.8533333539962769


In [69]:
# Get the optimal hyperparameters
# optimal_hps=tuner.get_best_hyperparameters(num_trials=1)[0]

# Get the best k sets of hyperparameters
best_k_hps = tuner.get_best_hyperparameters(num_trials=5)  # e.g., k = 5
for best_hp in best_k_hps:
  print(f"units: {best_hp.get('units_1')}; learning_rate: {best_hp.get('learning_rate')}")

units: 224; learning_rate: 0.081
units: 192; learning_rate: 0.081
units: 64; learning_rate: 0.009000000000000001
units: 32; learning_rate: 0.027
units: 96; learning_rate: 0.243


When search is over, you can retrieve the best model(s). The model is saved at its best performing epoch evaluated on the validation_data.

In [70]:
# Get the top 2 models.
models = tuner.get_best_models(num_models=2)
best_model = models[0]
# Build the model
# only needed for `Sequential` without specified `input_shape`.
# best_model.build(input_shape=(None, 28, 28))
best_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 224)            │       175,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224)            │           896 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 384)            │        86,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         3,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 266,986 (1.02 MB)

 Trainable params: 266,538 (1.02 MB)

 Non-trainable params: 448 (1.75 KB)

## Train the model



Find the optimal number of epochs to train the model with the hyperparameters obtained from the search.

In [71]:
# Build the model with the optimal hyperparameters and train it on the data for 50 epochs
model = model_builder(best_k_hps[0])

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 224)            │       175,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224)            │           896 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 384)            │        86,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │         3,850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 266,986 (1.02 MB)

 Trainable params: 266,538 (1.02 MB)

 Non-trainable params: 448 (1.75 KB)

In [72]:
history = model.fit(img_train, label_train, batch_size=256, epochs=50, validation_split=0.2)

val_acc_per_epoch = history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print('Best epoch: %d' % (best_epoch,))

Epoch 1/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.7554 - loss: 0.7196 - val_accuracy: 0.8203 - val_loss: 0.6245
Epoch 2/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8609 - loss: 0.3872 - val_accuracy: 0.8522 - val_loss: 0.4173
Epoch 3/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8771 - loss: 0.3370 - val_accuracy: 0.8637 - val_loss: 0.3795
Epoch 4/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8883 - loss: 0.3135 - val_accuracy: 0.8607 - val_loss: 0.3773
Epoch 5/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8938 - loss: 0.2898 - val_accuracy: 0.8718 - val_loss: 0.3530
Epoch 6/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8975 - loss: 0.2787 - val_accuracy: 0.8532 - val_loss: 0.4187
Epoch 7/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9063 - loss: 0.2600 - val_accuracy: 0.8784 - val_loss: 0.3353
Epoch 8/50
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9109 - loss: 0.2423 - val_accuracy: 0

To finish this tutorial, evaluate the hypermodel on the test data.

In [73]:
eval_result = model.evaluate(img_test, label_test)
print("[test loss, test accuracy]:", eval_result)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8745 - loss: 0.5094
[test loss, test accuracy]: [0.5036908984184265, 0.8716999888420105]


The `my_dir/intro_to_kt` directory contains detailed logs and checkpoints for every trial (model configuration) run during the hyperparameter search. If you re-run the hyperparameter search, the Keras Tuner uses the existing state from these logs to resume the search. To disable this behavior, pass an additional `overwrite=True` argument while instantiating the tuner.

<font color = 'steelblue'><b>Exercise</b> </font>

Q2: Calculate the total number of parameters for the model below and provide the calculation steps.

In [74]:
model_ex2 = Sequential([layers.Flatten(input_shape=[28, 28], name="flattened_input"),
                        layers.Dense(500, name="fully_connected_1"),
                        layers.BatchNormalization(),
                        layers.ReLU(),
                        layers.Dense(200, name="fully_connected_2"),
                        layers.BatchNormalization(),
                        layers.ReLU(),
                        layers.Dropout(0.3),
                        layers.Dense(10, name="output")],
                        name="my_mnist_model")


In [75]:
model_ex2.summary()

Model: "my_mnist_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flattened_input (Flatten)       │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fully_connected_1 (Dense)       │ (None, 500)            │       392,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 500)            │         2,000 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 500)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fully_connected_2 (Dense)       │ (None, 200)            │       100,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 200)            │           800 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         2,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 497,510 (1.90 MB)

 Trainable params: 496,110 (1.89 MB)

 Non-trainable params: 1,400 (5.47 KB)

Answer: (784x500+500) + 500x4 + (500x200+200) + 200x4 + (200x10+10) = 497510

## Summary

In this tutorial, you learned how to use the Keras Tuner to tune hyperparameters for a model. To learn more about the Keras Tuner, check out these additional resources:

* [Keras Tuner - Hyperparameters](https://keras.io/api/keras_tuner/hyperparameters/)
* [Keras Tuner website](https://keras-team.github.io/keras-tuner/)

Also check out the [HParams Dashboard](https://www.tensorflow.org/tensorboard/hyperparameter_tuning_with_hparams) in TensorBoard to interactively tune your model hyperparameters.